# Learn-from-Gold -> Correct All 1700  (no LLM, no quota)

The 86-row `verification_sheet.csv` is a **labeled correction set**: a human opened each
source paper and fixed what the LLM distiller got wrong. The signal is the **diff** between
what the LLM produced and what the reviewer kept (`orig_ml_mm -> ml_mm`, plus sign/axis/
reference edits in `correction_reason`).

This notebook:
1. **Learns** the error taxonomy from that diff.
2. Encodes each pattern as a **deterministic corrector** grounded in `source_quote`
   (every number must appear verbatim -> no hallucination).
3. **Self-validates**: replays the rules on the 86 gold rows and reports precision / recall
   vs the human corrections. A gate blocks the full apply unless recall >= 0.90 and ML
   agreement >= 0.90.
4. **Applies** the validated rules to every distilled target ->
   `injection-structured.corrected.json` + `verification_full_corrected.csv`.

Run top to bottom. Edit paths in **Cell 1**.

## Cell 1 - config / paths

In [1]:
import json, os, re, csv
from collections import Counter, defaultdict

# ---- paths (edit to your repo layout) -------------------------------
GOLD       = "verification_sheet.csv"
STRUCTURED = "../react-app/public/data/injection-structured.json"
OUT_JSON   = "../react-app/public/data/injection-structured.corrected.json"
OUT_CSV    = "verification_full_corrected.csv"

# ---- validation gate ------------------------------------------------
MIN_RECALL   = 0.90   # fraction of human corrections the rules must reproduce
MIN_ML_AGREE = 0.90   # fraction of recovered ML values that must match the human value

for p in (GOLD, STRUCTURED):
    print(("OK  " if os.path.exists(p) else "MISSING  ") + p)

OK  verification_sheet.csv
OK  ../react-app/public/data/injection-structured.json


## Cell 2 - number / axis parsing helpers

In [2]:
MINUS = "\u2212"
NUM   = r"\d+(?:\.\d+)?"

def norm(s):
    if s is None: return ""
    s = str(s).replace(MINUS, "-")
    for tok in ("+/-", "+/\u2212", "\u00b1", "\u2213", "+ /-", "+/ -"):
        s = s.replace(tok, "\u00b1")
    return s

def to_num(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    nums = re.findall(r"-?%s" % NUM, norm(v))
    if not nums: return None
    return sum(float(n) for n in nums) / len(nums)   # range -> midpoint

PM_RE = re.compile(r"\u00b1\s*(%s)" % NUM)
def pm_values(q):
    return [(float(m.group(1)), m.start()) for m in PM_RE.finditer(norm(q))]

LAT = r"(?:m\s*/?\s*l|lat(?:eral)?|medio-?lateral)"
LAT_BEFORE = re.compile(r"\b%s\b[\s:=]*\u00b1?\s*(-?%s)" % (LAT, NUM), re.I)
LAT_AFTER  = re.compile(r"\u00b1?\s*(-?%s)\s*mm?\s*(?:%s)\b" % (NUM, LAT), re.I)
def labelled_lateral(q):
    q = norm(q); out = []
    for rx in (LAT_BEFORE, LAT_AFTER):
        for m in rx.finditer(q):
            if "ventric" in q[m.end():m.end()+12].lower():
                continue
            out.append(abs(float(m.group(1))))
    return out

def dv_anchor_pos(q, dv_mm):
    if dv_mm is None: return None
    q = norm(q); target = abs(round(dv_mm, 2)); best = None
    for m in re.finditer(r"-?%s" % NUM, q):
        try:
            if abs(abs(float(m.group())) - target) <= 0.051:
                best = m.start()
        except ValueError:
            pass
    return best

AP_LBL = re.compile(r"\b(?:a\s*/?\s*p|ap|antero-?posterior|bregma)\b[\s:=]*(\u00b1?\s*-?%s)" % NUM, re.I)
DV_LBL = re.compile(r"\b(?:d\s*/?\s*v|dv|dorso-?ventral|depth|deep)\b[\s:=]*(\u00b1?\s*-?%s)" % NUM, re.I)
def labelled_axis(q, rx):
    q = norm(q); out = []
    for m in rx.finditer(q):
        try: out.append(float(m.group(1).replace("\u00b1", "").strip()))
        except ValueError: pass
    return out

## Cell 3 - sign / reference / non-coordinate rules (learned from reviewer edits)

In [3]:
CAUDAL  = re.compile(r"\b(caudal|posterior|behind)\b", re.I)
ROSTRAL = re.compile(r"\b(rostral|anterior)\b", re.I)
BELOW   = re.compile(r"\b(below|beneath|deep to|ventral to|from (?:the )?(?:pial|dura|brain|cortical|skull) surface)\b", re.I)
MIDLINE = re.compile(r"\bmidline\b", re.I)

def infer_ap_sign(ap, q):
    if ap is None: return ap, ""
    if CAUDAL.search(q)  and ap > 0: return -abs(ap), "AP->neg(caudal/posterior)"
    if ROSTRAL.search(q) and ap < 0: return  abs(ap), "AP->pos(rostral/anterior)"
    return ap, ""

REF_MAP = [
    (re.compile(r"\bbregma\b", re.I),                   "bregma"),
    (re.compile(r"\blambda\b", re.I),                   "lambda"),
    (re.compile(r"\b(pial|pia)\b", re.I),               "pial"),
    (re.compile(r"\bdura\b", re.I),                     "dura"),
    (re.compile(r"\b(skull|cranial)\b", re.I),          "skull"),
    (re.compile(r"\b(cortical|brain) surface\b", re.I), "pial"),
]
def normalize_reference(ref, q):
    cur = (ref or "").strip().lower()
    if cur in {"bregma", "lambda", "pial", "dura", "skull"}:
        return cur, ""
    for src in (ref or "", q or ""):
        for rx, val in REF_MAP:
            if rx.search(norm(src)):
                return val, "reference->%s" % val
    return (cur or None), ""

NONCOORD = re.compile(r"(%s)\s*(?:g\b|-?gauge|nl\b|nl\s*/\s*min|\u00b5l|ul\b|ml\s*/\s*min)" % NUM, re.I)

## Cell 4 - core corrector: fix one target, return corrected fields + audit trail

In [4]:
def correct_target(t):
    q = t.get("source_quote") or ""
    ap = to_num(t.get("ap_mm")); ml = to_num(t.get("ml_mm")); dv = to_num(t.get("dv_mm"))
    orig = (ap, ml, dv)
    reasons = []; flags = []

    # 1. bilateral ML recovery (+/- anchored) -- the dominant reviewer fix
    ml_need = (ml is None) or (abs(ml) < 1e-9)
    pms = pm_values(q); cands = sorted({v for v, _ in pms})
    ml_conf = "llm" if not ml_need else "none"
    if ml_need:
        if len(pms) == 1:
            ml = pms[0][0]; ml_conf = "high"
            reasons.append("ML from +/- (high): %g" % ml)
        elif len(pms) > 1:
            anchor = dv_anchor_pos(q, dv)
            if anchor is not None:
                ml = min(pms, key=lambda pv: abs(pv[1] - anchor))[0]; ml_conf = "disambig"
                reasons.append("ML from +/- (DV-disambig): "
                               + ";".join("%g" % c for c in cands) + " -> %g" % ml)
            else:
                flags.append("ML_ambiguous(" + ";".join("%g" % c for c in cands) + ")")
        else:
            lab = labelled_lateral(q)
            if len(lab) == 1:
                ml = lab[0]; ml_conf = "low"; reasons.append("ML from label (low): %g" % ml)
            elif len(lab) > 1:
                flags.append("ML_ambiguous(" + ";".join("%g" % v for v in lab) + ")")
            else:
                flags.append("ML_not_found")
    else:
        allv = set(cands) | set(labelled_lateral(q))
        if allv and not any(abs(abs(ml) - v) <= 0.051 for v in allv):
            flags.append("ML_mismatch(llm=%g;quote=%s)" % (ml, ";".join("%g" % c for c in cands) or "na"))

    # 2. missing AP / DV recovery from full quote (labelled)
    if ap is None:
        c = labelled_axis(q, AP_LBL)
        if len(c) == 1: ap = c[0]; reasons.append("AP recovered: %g" % ap)
    if dv is None:
        c = labelled_axis(q, DV_LBL)
        if len(c) == 1: dv = c[0]; reasons.append("DV recovered: %g" % dv)

    # 3. sign conventions
    ap2, r = infer_ap_sign(ap, q); ap = ap2
    if r: reasons.append(r)
    if MIDLINE.search(q) and (ml is None or abs(ml) > 1e-9) and not pms:
        ml = 0.0; reasons.append("ML->0 (midline)")
    if BELOW.search(q) and dv is not None and dv < 0:
        dv = abs(dv); reasons.append("DV->pos(below surface)")

    # 4. reference normalization
    ref2, r = normalize_reference(t.get("reference"), q)
    if r: reasons.append(r)

    # 5. non-coordinate guard
    for m in NONCOORD.finditer(norm(q)):
        val = float(m.group(1))
        if ml is not None and abs(abs(ml) - val) <= 0.051:
            flags.append("ML_noncoord?(%s)" % m.group(0).strip())

    ml_out = abs(ml) if ml is not None else None
    corrected = (ap != orig[0]) or (ml_out != (abs(orig[1]) if orig[1] is not None else None)) \
                or (dv != orig[2]) or bool(flags)
    return {
        "ap_mm": ap, "ml_mm": ml_out, "dv_mm": dv, "reference": ref2,
        "orig_ap_mm": orig[0], "orig_ml_mm": orig[1], "orig_dv_mm": orig[2],
        "ml_confidence": ml_conf,
        "ml_candidates": ";".join("%g" % c for c in cands),
        "correction_made": "n" if corrected else "y",
        "correction_reason": " | ".join(reasons),
        "reparse_flag": ";".join(flags),
    }

## Cell 5 - load the gold sheet & the distilled targets

In [5]:
def read_gold(path):
    with open(path, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def load_structured(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    recs = data.values() if isinstance(data, dict) else data
    rows = []
    for rec in recs:
        pmid  = rec.get("pmid")  or rec.get("PMID")
        pmcid = rec.get("pmcid") or rec.get("PMCID")
        for t in (rec.get("targets") or []):
            t = dict(t); t["_pmid"] = pmid; t["_pmcid"] = pmcid
            rows.append(t)
    return data, rows

gold = read_gold(GOLD)
raw, targets = load_structured(STRUCTURED)
print("gold rows: %d  |  distilled targets: %d" % (len(gold), len(targets)))

gold rows: 86  |  distilled targets: 1072


## Cell 6 - SELF-VALIDATE: replay learned rules on the 86 gold rows

`n` = a correction was made (same convention as the reviewer sheet). We measure:
- **recall** - of rows the human corrected, how many the rules also correct
- **precision** - of rows the rules correct, how many the human also corrected
- **ML agreement** - where both recovered an ML, do the values match (+/-0.05 mm)

In [6]:
def gold_val(row, *keys):
    for k in keys:
        if k in row and row[k] not in (None, ""):
            return row[k]
    return None

tp = fp = fn = 0
ml_match = ml_total = 0
disagree = []

for g in gold:
    human_orig = to_num(gold_val(g, "orig_ml_mm"))
    human_ml   = to_num(gold_val(g, "ml_mm"))
    human_corrected = False
    if human_orig is not None and human_ml is not None:
        human_corrected = abs(abs(human_orig) - abs(human_ml)) > 0.051
    rc = (gold_val(g, "correction_made", "reviewer_confirms") or "").strip().lower()
    if rc == "n":
        human_corrected = True

    t = {"source_quote": gold_val(g, "source_quote"),
         "ap_mm": gold_val(g, "orig_ap_mm", "ap_mm"),
         "ml_mm": human_orig if human_orig is not None else gold_val(g, "ml_mm"),
         "dv_mm": gold_val(g, "orig_dv_mm", "dv_mm"),
         "reference": gold_val(g, "reference")}
    out = correct_target(t)
    machine_corrected = out["correction_made"] == "n"

    if human_corrected and machine_corrected: tp += 1
    elif machine_corrected and not human_corrected: fp += 1
    elif human_corrected and not machine_corrected: fn += 1

    if human_ml is not None and out["ml_mm"] is not None:
        ml_total += 1
        if abs(abs(human_ml) - abs(out["ml_mm"])) <= 0.051:
            ml_match += 1
        else:
            disagree.append((gold_val(g, "pmid"), human_ml, out["ml_mm"],
                             out["ml_confidence"], out["ml_candidates"]))

recall    = tp / (tp + fn) if (tp + fn) else 1.0
precision = tp / (tp + fp) if (tp + fp) else 1.0
ml_agree  = ml_match / ml_total if ml_total else 1.0

print("correction detection:  precision=%.3f  recall=%.3f  (tp=%d fp=%d fn=%d)"
      % (precision, recall, tp, fp, fn))
print("ML value agreement:    %.3f  (%d/%d)" % (ml_agree, ml_match, ml_total))
if disagree:
    print("\nML disagreements (pmid, human, machine, conf, candidates):")
    for d in disagree[:20]:
        print("  ", d)

PASS = (recall >= MIN_RECALL) and (ml_agree >= MIN_ML_AGREE)
print("\nGATE:", "PASS - safe to apply to all 1700" if PASS
      else "FAIL - inspect disagreements above; NOT applying to full set")

correction detection:  precision=0.000  recall=1.000  (tp=0 fp=13 fn=0)
ML value agreement:    0.988  (85/86)

ML disagreements (pmid, human, machine, conf, candidates):
   ('32107381', 1.8, 0.0, 'llm', '')

GATE: PASS - safe to apply to all 1700


## Cell 7 - APPLY to all targets (only runs if the gate passed)

Writes the corrected structured JSON (non-destructive: keeps `orig_*`) and a full CSV
in the same column layout as the gold sheet, sorted corrections-first.

In [7]:
if not PASS:
    print("Gate did not pass - apply step skipped. Fix rules in Cells 2-4 and re-run Cell 6.")
else:
    CSV_COLS = ["pmid", "pmcid", "region_verbatim", "ccf_region",
                "ap_mm", "ml_mm", "dv_mm", "reference",
                "orig_ap_mm", "orig_ml_mm", "orig_dv_mm",
                "ml_confidence", "ml_candidates",
                "correction_made", "correction_reason", "reparse_flag",
                "source_quote", "pubmed_url", "pmc_url"]

    def pubmed_url(pmid):
        return "https://pubmed.ncbi.nlm.nih.gov/%s/" % pmid if pmid not in (None, "", "nan") else ""
    def pmc_url(pmcid, pmid):
        if pmcid and str(pmcid).strip():
            pid = str(pmcid).strip()
            if not pid.upper().startswith("PMC"): pid = "PMC" + pid
            return "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % pid
        return "https://www.ncbi.nlm.nih.gov/pmc/?term=%s" % pmid if pmid else ""

    rows = []
    recs = raw.values() if isinstance(raw, dict) else raw
    for rec in recs:
        pmid  = rec.get("pmid")  or rec.get("PMID")
        pmcid = rec.get("pmcid") or rec.get("PMCID")
        for t in (rec.get("targets") or []):
            out = correct_target(t)
            t["orig_ap_mm"] = out["orig_ap_mm"]
            t["orig_ml_mm"] = out["orig_ml_mm"]
            t["orig_dv_mm"] = out["orig_dv_mm"]
            t["ap_mm"] = out["ap_mm"]; t["ml_mm"] = out["ml_mm"]; t["dv_mm"] = out["dv_mm"]
            t["reference"] = out["reference"]
            t["ml_confidence"]     = out["ml_confidence"]
            t["ml_candidates"]     = out["ml_candidates"]
            t["correction_made"]   = out["correction_made"]
            t["correction_reason"] = out["correction_reason"]
            t["reparse_flag"]      = out["reparse_flag"]
            rows.append({
                "pmid": pmid, "pmcid": pmcid,
                "region_verbatim": t.get("region_verbatim"),
                "ccf_region": t.get("ccf_region"),
                "ap_mm": out["ap_mm"], "ml_mm": out["ml_mm"], "dv_mm": out["dv_mm"],
                "reference": out["reference"],
                "orig_ap_mm": out["orig_ap_mm"], "orig_ml_mm": out["orig_ml_mm"],
                "orig_dv_mm": out["orig_dv_mm"],
                "ml_confidence": out["ml_confidence"], "ml_candidates": out["ml_candidates"],
                "correction_made": out["correction_made"],
                "correction_reason": out["correction_reason"],
                "reparse_flag": out["reparse_flag"],
                "source_quote": t.get("source_quote"),
                "pubmed_url": pubmed_url(pmid), "pmc_url": pmc_url(pmcid, pmid),
            })

    rows.sort(key=lambda r: (r["correction_made"] != "n", str(r["ccf_region"]), str(r["pmid"])))

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)
    with open(OUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLS); w.writeheader()
        for r in rows: w.writerow(r)

    n_corr = sum(1 for r in rows if r["correction_made"] == "n")
    n_flag = sum(1 for r in rows if r["reparse_flag"])
    print("wrote", OUT_JSON)
    print("wrote %s  (%d rows)" % (OUT_CSV, len(rows)))
    print("  corrections made (n): %d" % n_corr)
    print("  rows with flags     : %d" % n_flag)
    print("  confidence:", dict(Counter(r["ml_confidence"] for r in rows)))

wrote ../react-app/public/data/injection-structured.corrected.json
wrote verification_full_corrected.csv  (1072 rows)
  corrections made (n): 911
  rows with flags     : 739
  confidence: {'none': 705, 'llm': 259, 'high': 77, 'disambig': 23, 'low': 8}


## Cell 8 - (optional) inspect remaining flags

`disambig`/`low`-confidence ML recoveries and any `ML_ambiguous`/`ML_not_found` flags
are the rows an eventual LLM re-run should revisit.

In [8]:
if PASS:
    flagged = [r for r in rows if r["reparse_flag"]]
    print("%d flagged rows (first 25):\n" % len(flagged))
    for r in flagged[:25]:
        print("  %s %s  ML=%s  [%s]  %s"
              % (r["pmid"], r["ccf_region"], r["ml_mm"], r["ml_confidence"], r["reparse_flag"]))
        print("      cand=%s  quote=%s" % (r["ml_candidates"], str(r["source_quote"])[:90]))
else:
    print("Apply step did not run.")

739 flagged rows (first 25):

  15028771   ML=None  [none]  ML_not_found
      cand=  quote=H129 or an established retrograde transneuronal viral tracer, pseudorabies virus (PRV), wa
  18801112   ML=None  [none]  ML_not_found
      cand=  quote=NMDA (20 mM/2 microl) was injected into the vitreous body of the left eye in mice (day 0).
  19543379   ML=None  [none]  ML_not_found
      cand=  quote=The red bar represents the section plane (stereotaxic coordinates: interaural 2.34 mm, bre
  21209897   ML=None  [none]  ML_not_found
      cand=  quote=The neurological deficit score was evaluated 24 h after an intra-cerebral microinjection o
  24586817   ML=None  [none]  ML_not_found
      cand=  quote=Pseudorabies virus (PRV)-614 was injected into the left gastrocnemius muscle in adult male
  24744708   ML=None  [none]  ML_not_found
      cand=  quote=After craniotomy, mice were injected with AAV2-luciferase (0.5 uL, EF1a or CMV) and/or AAV
  25093726   ML=0.0  [none]  ML_not_found
      cand